# Vagues de chaleur marines en Méditerranée — tutoriel MHEAT

## Contexte scientifique

Une **Vague de Chaleur Marine (VCM)**, ou *Marine Heatwave* (MHW), est un événement océanique discret, prolongé et anormalement chaud. Formellement, Hobday *et al.* (2016) définissent une VCM comme une période d'au moins **cinq jours consécutifs** pendant laquelle la température de surface de la mer (SST) dépasse le 90ᵉ centile variable selon la saison, calculé sur une climatologie de référence de 30 ans (typiquement 1991–2020). Une fois détecté, l'événement est classé en cinq catégories de sévérité (I Modérée → V Super-Extrême) à partir du rapport entre l'anomalie maximale et l'anomalie seuil. Parce que la définition est relative à la climatologie locale, la même température absolue peut constituer une VCM dans une région et rester parfaitement normale dans une autre.

La **mer Méditerranée** est l'un des bassins océaniques qui se réchauffent le plus vite : la température de surface a augmenté d'environ 0,4 °C par décennie depuis les années 1980, soit à peu près trois fois la moyenne mondiale, et le bassin a connu des VCM à l'échelle du bassin entier en 2003, 2015, 2017, 2018, 2022 et 2023. Ces événements coïncident avec des mortalités massives d'invertébrés benthiques (gorgones, éponges, bivalves), des pertes en aquaculture de poissons, et un blanchissement à grande échelle des herbiers de *Posidonia oceanica* — un habitat-clé méditerranéen. Quantifier la co-localisation des événements extrêmes avec les secteurs vulnérables est un besoin politique direct au titre de la Directive-cadre Stratégie pour le milieu marin et de la Stratégie Biodiversité 2030 de l'UE.

Ce notebook parcourt le flux de travail MHEAT sur un **cube SST synthétique fourni** afin que vous puissiez exécuter chaque cellule **sans compte Copernicus** ni accès Internet. Le même code, inchangé, fonctionne sur les données réelles du Copernicus Marine Service une fois les identifiants fournis — voir la dernière section.

---

## Dépendances

```bash
pip install xarray numpy pandas matplotlib marineHeatWaves netCDF4
# Optionnel, pour la récupération en direct depuis CMS :
pip install copernicusmarine
```

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# Make the MHEAT backend importable. Adds any of (repo/backend, /srv/app)
# depending on whether the notebook is run from a clone or inside the
# Docker image.
for candidate in (os.path.abspath('../backend'), '/srv/app'):
    if os.path.isdir(os.path.join(candidate, 'app')) and candidate not in sys.path:
        sys.path.insert(0, candidate)
print('Python', sys.version.split()[0])

## Étape 1 — Charger un cube SST méditerranéen

Nous utilisons le cube synthétique livré avec MHEAT. C'est un champ SST quotidien de 3 ans (2020–2022) à 0,25° couvrant la plus grande partie de la Méditerranée occidentale et centrale, avec un cycle saisonnier réaliste et une forte anomalie chaude injectée en juillet–août 2022 pour imiter la vague de chaleur historique de cet été-là.

Le cube a la forme ``(1096 jours × 41 lat × 73 lon)``.

In [ ]:
from app.sst import build_synthetic_med_cube

ds = build_synthetic_med_cube(anomaly_year=2022)
print(ds)

## Étape 2 — Climatologie saisonnière + seuil au 90ᵉ centile

Nous choisissons le pixel le plus chaud pour l'illustration, exécutons ``marineHeatWaves.detect()`` et traçons les trois courbes de référence : la SST quotidienne, la moyenne lissée par jour de l'année (climatologie) et le seuil au 90ᵉ centile. Les zones rouges ombrées sont les VCM identifiées par le détecteur.

In [ ]:
from marineHeatWaves import detect as mhw_detect

da = ds['analysed_sst']

# Warmest pixel at the 2022 peak day.
peak_day = da.sel(time='2022-07-25').mean(['latitude', 'longitude']).item()
# Actually choose the hottest pixel on that date.
peak_slice = da.sel(time='2022-07-25')
iy, ix = np.unravel_index(int(np.argmax(peak_slice.values)), peak_slice.shape)
lat_val = float(da['latitude'][iy]); lon_val = float(da['longitude'][ix])
print(f'Hottest pixel at 2022-07-25: lat={lat_val:.2f}° lon={lon_val:.2f}°')

series = da.isel(latitude=iy, longitude=ix).values.astype('float64')
times = pd.to_datetime(da.time.values)
ordinals = np.array([t.toordinal() for t in times])

mhws, clim = mhw_detect(
    ordinals, series,
    climatologyPeriod=[times[0].year, times[-1].year - 1],
    pctile=90, windowHalfWidth=5,
    minDuration=5, joinAcrossGaps=True, maxGap=2,
)
print(f"Events detected on this pixel: {mhws['n_events']}")
for i in range(mhws['n_events']):
    s = pd.Timestamp.fromordinal(int(mhws['time_start'][i]))
    e = pd.Timestamp.fromordinal(int(mhws['time_end'][i]))
    print(f"  #{i+1}: {s.date()} → {e.date()}  duration={int(mhws['duration'][i])}d  i_max={mhws['intensity_max'][i]:.2f}°C")

In [ ]:
# Plot 1 — time series with threshold bands
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(times, series, label='SST', color='black', lw=0.8)
ax.plot(times, clim['seas'], label='Climatology (DOY mean)', color='tab:blue')
ax.plot(times, clim['thresh'], label='90th percentile', color='tab:orange', ls='--')
for i in range(mhws['n_events']):
    s = pd.Timestamp.fromordinal(int(mhws['time_start'][i]))
    e = pd.Timestamp.fromordinal(int(mhws['time_end'][i]))
    ax.axvspan(s, e, color='red', alpha=0.25)
ax.set_ylabel('SST (°C)')
ax.set_title(f'Hobday MHW detection — pixel ({lat_val:.2f}°N, {lon_val:.2f}°E)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## Étape 3 — Détection pixel par pixel + regroupement avec les utilitaires MHEAT

Le backend encapsule ``marineHeatWaves.detect`` dans ``detect_cube`` (pixel par pixel) et ``cluster_events`` (fusionne les voisins spatio-temporels). Ensemble ils transforment des milliers de minuscules événements par pixel en quelques polygones de région contigus — l'unité d'analyse pertinente pour les études d'impact.

In [ ]:
from app.mhw import detect_cube, cluster_events

events = detect_cube(da, clim_period=(2020, 2021), max_pixels=400)
print(f'Per-pixel events detected: {len(events)}')

clusters = cluster_events(events)
print(f'After clustering: {len(clusters)}')
for c in clusters[:5]:
    print(f"  {c.event_id}  {c.category_name}  {c.date_start} → {c.date_end}  n_pixels={c.n_pixels}")

## Étape 4 — Carte spatiale de la densité d'événements + histogramme d'intensité

In [ ]:
# Plot 2 — spatial map of per-pixel event counts, derived from the detected events
grid = np.zeros((da.sizes['latitude'], da.sizes['longitude']))
lat_vals = da['latitude'].values
lon_vals = da['longitude'].values
for e in events:
    iy = int(np.argmin(np.abs(lat_vals - e.centroid_lat)))
    ix = int(np.argmin(np.abs(lon_vals - e.centroid_lon)))
    grid[iy, ix] += 1

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.pcolormesh(lon_vals, lat_vals, grid, cmap='YlOrRd', shading='auto')
plt.colorbar(im, ax=ax, label='# events at pixel')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Per-pixel MHW event count — 2020–2022 synthetic Mediterranean')
# Overlay cluster centroids as black stars
for c in clusters:
    ax.plot(c.centroid_lon, c.centroid_lat, marker='*', color='black', ms=12, mew=0.5)
plt.tight_layout(); plt.show()

In [ ]:
# Plot 3 — intensity histogram by Hobday category
fig, ax = plt.subplots(figsize=(8, 4))
intens = [e.intensity_max for e in events]
cats = [e.category for e in events]
cat_colors = {1:'#ffd166', 2:'#ff9f1c', 3:'#e63946', 4:'#9d0208', 5:'#370617'}
cat_labels = {1:'I Moderate', 2:'II Strong', 3:'III Severe', 4:'IV Extreme', 5:'V Super-Extreme'}
bins = np.linspace(0, max(intens) + 0.5, 20)
for c in sorted(set(cats)):
    sub = [i for i, cc in zip(intens, cats) if cc == c]
    ax.hist(sub, bins=bins, alpha=0.7, label=cat_labels[c], color=cat_colors[c])
ax.set_xlabel('Peak intensity (°C above threshold)')
ax.set_ylabel('# events')
ax.set_title('Distribution of per-pixel MHW peak intensities')
ax.legend()
plt.tight_layout(); plt.show()

## Étape 5 — Jointure avec les couches sectorielles

Le backend embarque un petit jeu de sites aquacoles méditerranéens, d'aires marines protégées Natura 2000 et de polygones d'herbiers sous ``backend/app/fixtures/overlays/``. ``attach_impact_properties`` signale, pour chaque événement, combien de sites aquacoles sont touchés et combien de km² d'habitat AMP / herbier le polygone d'événement couvre.

In [ ]:
import json
from pathlib import Path
from app.impact import attach_impact_properties
from app.mhw import events_to_geojson

# Find the overlay fixtures dir — either in the cloned repo or inside the container.
candidates = [Path('../backend/app/fixtures/overlays'), Path('/srv/app/app/fixtures/overlays')]
fixtures_dir = next((p for p in candidates if p.is_dir()), candidates[0])

overlays = {k: json.loads((fixtures_dir / f'{k}.json').read_text()) for k in ('aquaculture','mpa','seagrass')}

geojson = events_to_geojson(clusters)
attach_impact_properties(geojson, clusters, overlays)
for f in geojson['features'][:3]:
    p = f['properties']
    imp = p.get('impact', {})
    print(f"{f['id']}  {p['category_name']}  aquaculture={imp.get('n_aquaculture_sites')}  mpa_km2={imp.get('mpa_area_km2')}  seagrass_km2={imp.get('seagrass_area_km2')}")

## Comment exécuter ce notebook sur EDITO Datalab

1. Dans l'EDITO Datalab, lancez le service **JupyterLab** (n'importe quelle image Python 3.11).
2. Ouvrez un terminal et `git clone https://github.com/<your-org>/mheat.git`, puis `cd mheat/tutorials` et ouvrez ce notebook.
3. `pip install -r ../backend/requirements.txt` depuis l'environnement du notebook (ou intégrez les dépendances dans votre image).
4. Pour exécuter avec les **vraies** données Copernicus Marine, définissez `COPERNICUSMARINE_SERVICE_USERNAME` et `COPERNICUSMARINE_SERVICE_PASSWORD` dans l'environnement du notebook et remplacez le chargement synthétique par :

```python
import copernicusmarine
out = copernicusmarine.subset(
    dataset_id='SST_MED_SST_L4_NRT_OBSERVATIONS_010_004',
    minimum_longitude=-6, maximum_longitude=36.5,
    minimum_latitude=30, maximum_latitude=46,
    start_datetime='2022-05-01T00:00:00', end_datetime='2022-09-30T23:59:59',
    variables=['analysed_sst'],
)
ds = xr.open_dataset(out)
```

5. Le reste du notebook s'exécute sans modification. Le service MHEAT lui-même (``docker compose up --build``) est également redéployable sur une instance EDITO **Onyxia** — pointez `FRONTEND_DIR` vers le bundle Vite compilé et exposez le port 8000.

## Citations

* Hobday, A. J., Alexander, L. V., Perkins, S. E., Smale, D. A., *et al.* (2016). *A hierarchical approach to defining marine heatwaves.* **Progress in Oceanography**, 141, 227–238.
* Hobday, A. J., Oliver, E. C. J., *et al.* (2018). *Categorizing and naming marine heatwaves.* **Oceanography**, 31(2), 162–173.
* Oliver, E. C. J. (2019). `marineHeatWaves` — paquet Python pour l'identification des vagues de chaleur marines. https://github.com/ecjoliver/marineHeatWaves
* Copernicus Marine Service — https://marine.copernicus.eu